### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# Quantization-Aware Training: CNN for Rain-Drop Classification

This notebook trains a small CNN to classify clear versus raindrop images with quantization-aware training (QAT). It prepares the Raindrop Clarity data, applies staged observer and fake-quantization training, validates the exported model, converts it for embedded deployment, and compares inference timing across hardware targets. QAT is evaluated as a deployment trade-off against the floating-point model and the post-training quantization workflow documented separately.

## Notebook Structure

1. **Import libraries and helper functions**
2. **Prepare the Raindrop Clarity data and inputs**
3. **Create the CNN model**
4. **Train with staged quantization-aware training**
5. **Generate representative inputs and export the QAT model**
6. **Validate the exported quantized model**
7. **Convert the QAT model for embedded deployment**
8. **Display inference timing and profiling results**
9. **Compare performance across hardware platforms**
10. **References and Generated Artifacts**

## 1. Import libraries and helper functions

Import the model, dataset, quantization, conversion, and benchmarking helpers used by the QAT workflow.

In [ ]:
import os
import sys

import modelling_helper as mh

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
import _CentralScripts.quantization_helper as qh
import _CentralScripts.mobilenet_helper as mn
import MobileNetV3ClassificationRainDrops.modelling_helper as mm

## 2. Prepare the Raindrop Clarity data and inputs

The notebook downloads and restructures the [Raindrop Clarity dataset](https://github.com/jinyeying/RaindropClarity?tab=readme-ov-file) [2]. The source dataset was designed for algorithmic raindrop removal, but this example repurposes its images for binary classification of clear and raindrop conditions. The helper creates train, validation, and test splits and applies the configured 100-pixel resolution and preprocessing.

Training data are used for QAT, validation data are used for model-quality checks, and representative tensors are used for export and conversion checks. Because the source data were not collected specifically for classification, class balance, camera variation, weather coverage, and label assumptions should be reviewed before deployment.

In [ ]:
batch_size = 16
resolution = 100

path = mm.fetch_sort_data()
dataloader, class_names, dataset_sizes = mn.get_dataloader(
    os.path.join(path, "train"), batch_size, mode="train", resolution=resolution
)
_, input_size = mm.get_input_dataloader(dataloader, is_plot=True)

## 3. Create the CNN model

Create the floating-point CNN that will be trained and then prepared for quantization-aware training. The model predicts the two retained classes: clear and raindrop.

In [ ]:
model = mh.create_model(input_size)

## 4. Train with staged quantization-aware training

The `torch.ao` QAT workflow trains the model while simulating deployment quantization. It uses three stabilization phases:

- **Observer phase:** observers track activation ranges and estimate scale and zero-point parameters while the model learns.
- **Fake-quantization phase:** observers are disabled and fake quantization remains active, exposing the model to quantization effects during training.
- **Frozen batch-normalization phase:** batch-normalization statistics are frozen later in training to stabilize the final quantized behavior.

The demonstration uses a short training run; meaningful accuracy comparisons require a representative schedule and fixed evaluation protocol.

In [ ]:
model = mh.train_quantize_model(
    model,
    dataloader,
    dataset_sizes,
    num_epochs=1,
    observer_epochs=3,
    freeze_bn_epoch=6,
)

## 5. Generate representative inputs and export the QAT model

Select a representative transformed input and compute its expected QAT output. The input/output pair defines the contract used to validate conversion and embedded execution.

In [ ]:
input_target, _ = mm.get_input_dataloader(dataloader, is_plot=True)
output_target = cs.get_predictions("torch", model, input_target)

The trained model is exported as `cnn_weather_qat` together with representative input and expected output artifacts. These files are consumed by the conversion and benchmarking stages.

In [ ]:
model_name = "cnn_weather_qat"
cs.save_all(model_name, input_target, output_target, model, origin="torch")

## 6. Validate the exported quantized model

Validate the exported QAT model on the held-out validation split. Compare classification behavior and output fidelity with the floating-point baseline and with the PTQ example where available; QAT is useful only when its quantization robustness justifies its additional training cost.

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)
model_paths = (onnx_model_file,)

dataloader_val, class_names, dataset_sizes = mn.get_dataloader(
    os.path.join(path, "val"), batch_size=1, mode="val", resolution=resolution
)

qh.validate_quantization_options(model_paths, dataloader_val)

## 7. Convert the QAT model for embedded deployment

Prepare the conversion service and convert the exported QAT model for AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4. Generated source, binaries, logs, timing, and profile results are stored below `out/cnn_weather_qat/test_cnn_weather_qat/<target>/`.

In [ ]:
cs.ensure_docker_container()

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)

target = "tc4dx_ppu"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m4"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
)
tool.convert_model()

## 8. Display inference timing and profiling results

Summarize the selected target's `results.json` timing section, including estimated cycles, instructions, average CPI, latency, and throughput. Profile shares identify operator contributions to execution cost and are separate from classification accuracy.

In [ ]:
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

## 9. Compare performance across hardware platforms

Run the comparison target with the same exported QAT model and representative data, then display results for AURIX™ and TRAVEO™ platforms: AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4. Compare latency and throughput alongside validation accuracy and output fidelity.

In [ ]:
target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
)
tool.convert_model()

In [ ]:
df = cs.display_platform_comparison(model_folder)

## 10. References and Generated Artifacts

- **QAT reference:** [PyTorch Quantization-Aware Training](https://pytorch.org/blog/quantization-aware-training/) [1].
- **Dataset:** [Raindrop Clarity](https://github.com/jinyeying/RaindropClarity?tab=readme-ov-file), Jin et al. (2024), originally created for day/night raindrop removal and repurposed here for clear-versus-raindrop classification [2].
- **Generated model artifacts:** `cnn_weather_qat` ONNX model, representative transformed input, expected output, and validation outputs.
- **Deployment artifacts:** target-specific source, binaries, logs, timing, profiling, and comparison reports under `out/cnn_weather_qat/` and `out/cnn_weather_qat/test_cnn_weather_qat/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.
- **Interpretation:** compare QAT accuracy, latency, model size, output fidelity, and conversion viability with the floating-point and PTQ baselines; results are limited by the source dataset and short demonstration training run.